# Vector Stores

Preparation

In [7]:
company_benefits_documents = [
    {
        "id": "benefit_001",
        "title": "Health and Wellness Program",
        "content": """Our Health and Wellness Program includes comprehensive health insurance, 
        dental and vision coverage, mental health support, and access to wellness resources. 
        Employees can also participate in fitness classes, receive gym membership discounts, 
        and join our annual wellness retreat. Coverage starts from day one of employment.""",
        "metadata": {"department": "hr", "type": "benefit"}
    },
    {
        "id": "benefit_002",
        "title": "Professional Development Fund",
        "content": """We offer a Professional Development Fund of $2,000 per year for each employee. 
        This fund can be used for attending conferences, enrolling in courses, obtaining certifications, 
        and purchasing educational materials. Employees are encouraged to discuss their development 
        plans with their managers to align with career goals.""",
        "metadata": {"department": "hr", "type": "benefit"}
    },
    {
        "id": "benefit_003",
        "title": "Flexible Work Hours",
        "content": """Our Flexible Work Hours policy allows employees to adjust their work schedules 
        to better fit their personal lives. Core working hours are from 10 AM to 3 PM, but employees 
        can choose to start earlier or finish later as long as they complete their required hours. 
        This policy aims to promote work-life balance and accommodate different lifestyles.""",
        "metadata": {"department": "hr", "type": "benefit"}
    },
    {
        "id": "benefit_004",
        "title": "Employee Stock Purchase Plan",
        "content": """The Employee Stock Purchase Plan (ESPP) allows employees to purchase company 
        stock at a discounted rate. Employees can contribute up to 10% of their salary to buy shares 
        at a 15% discount. This plan is designed to encourage long-term investment in the company 
        and align employee interests with company success.""",
        "metadata": {"department": "finance", "type": "benefit"}
    }
]


## Chroma

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma_client = chromadb.Client()

In [ ]:
# Create a new collection
collection = chroma_client.create_collection("company_benefits")

# Set up the embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Add documents to the collection
for doc in company_benefits_documents:
    embeddings = embedding_model.encode(doc["content"])
    collection.add(
        documents=[doc["content"]],
        metadatas=[doc["metadata"]],
        embeddings=[embeddings],
        ids=[doc["id"]]
    )

In [13]:
query_embedding = embedding_model.encode("stock purchase").tolist()
results = collection.query(query_embeddings=[query_embedding], n_results=2)
print(results)

{'ids': [['benefit_004', 'benefit_002']], 'embeddings': None, 'documents': [['The Employee Stock Purchase Plan (ESPP) allows employees to purchase company \n        stock at a discounted rate. Employees can contribute up to 10% of their salary to buy shares \n        at a 15% discount. This plan is designed to encourage long-term investment in the company \n        and align employee interests with company success.', 'We offer a Professional Development Fund of $2,000 per year for each employee. \n        This fund can be used for attending conferences, enrolling in courses, obtaining certifications, \n        and purchasing educational materials. Employees are encouraged to discuss their development \n        plans with their managers to align with career goals.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'department': 'finance', 'type': 'benefit'}, {'type': 'benefit', 'department': 'hr'}]], 'distances': [[0.8917592763900757, 1.

In [ ]:
# Persistent client
# storage_path = 

# persistent_client = chromadb.PersistentClient(path=storage_path)

# collection = persistent_client.get_or_create_collection(name="localcollection")

# if collection.count() < 0:
#     collection.add(
#         documents=[doc["content"] for doc in company_benefits_documents],
#         metadatas=[doc["metadata"] for doc in company_benefits_documents],
#         embeddings=[embedding_model.encode(doc["content"]).tolist() for doc in company_benefits_documents],
#         ids=[doc["id"] for doc in company_benefits_documents]
# )

## Faiss

In [14]:
import numpy as np
import faiss

In [17]:
document_embeddings = np.array([embedding_model.encode(doc["content"]) for doc in company_benefits_documents])
document_embeddings

array([[-0.00606633,  0.0828816 ,  0.05840454, ..., -0.05294187,
         0.05025507, -0.02304508],
       [ 0.07172127,  0.02182724, -0.03397663, ..., -0.06312494,
         0.06198026, -0.00032927],
       [-0.06466026,  0.01105874, -0.0076797 , ...,  0.00874641,
        -0.03561554, -0.00232708],
       [-0.00453231,  0.0477905 ,  0.04850495, ..., -0.03419175,
         0.02681308,  0.01554039]], shape=(4, 384), dtype=float32)

In [20]:
# Create FAISS index
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(document_embeddings))

# Search
query = embedding_model.encode(["stock purchase"])
D, I = index.search(np.array(query), k=2)
for i, idx in enumerate(I[0]):
    print(f"{i + 1}: {company_benefits_documents[idx]} (Distance: {D[0][i]})")

1: {'id': 'benefit_004', 'title': 'Employee Stock Purchase Plan', 'content': 'The Employee Stock Purchase Plan (ESPP) allows employees to purchase company \n        stock at a discounted rate. Employees can contribute up to 10% of their salary to buy shares \n        at a 15% discount. This plan is designed to encourage long-term investment in the company \n        and align employee interests with company success.', 'metadata': {'department': 'finance', 'type': 'benefit'}} (Distance: 0.8917592167854309)
2: {'id': 'benefit_002', 'title': 'Professional Development Fund', 'content': 'We offer a Professional Development Fund of $2,000 per year for each employee. \n        This fund can be used for attending conferences, enrolling in courses, obtaining certifications, \n        and purchasing educational materials. Employees are encouraged to discuss their development \n        plans with their managers to align with career goals.', 'metadata': {'department': 'hr', 'type': 'benefit'}} (Dis

## Qdrant

In [25]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(":memory:")

In [26]:
client.create_collection(
    collection_name="company_benefits",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

True

In [28]:
collection_info = client.get_collection(collection_name="company_benefits")
list(collection_info)

[('status', <CollectionStatus.GREEN: 'green'>),
 ('optimizer_status', <OptimizersStatusOneOf.OK: 'ok'>),
 ('vectors_count', None),
 ('indexed_vectors_count', 0),
 ('points_count', 0),
 ('segments_count', 1),
 ('config',
  CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1), wal_config=WalConfig(wal_capacity_mb

In [30]:
doc_texts = [doc["content"] for doc in company_benefits_documents]
# Prepare points to insert into Qdrant
points = [
    PointStruct(id=i, vector=embedding, payload={"text": doc})
    for i, (embedding, doc) in enumerate(zip(document_embeddings, doc_texts))
]

points

[PointStruct(id=0, vector=[-0.006066326051950455, 0.08288159966468811, 0.05840454250574112, 0.020230470225214958, -0.008119749836623669, 0.08219286799430847, 0.010208413936197758, -0.11232389509677887, -0.0976911261677742, -0.0775367021560669, 0.016001636162400246, 0.03081212379038334, -0.00836178008466959, -0.07698342949151993, 0.09790770709514618, -0.028921188786625862, 0.05730446055531502, 0.06191138178110123, -0.0029255750123411417, -0.021226003766059875, -0.11525201052427292, 0.017889143899083138, 0.0009007732733152807, 0.06961091607809067, -0.07721550762653351, 0.07233696430921555, 0.002220580354332924, -0.007128901779651642, -0.05104753375053406, 0.03714298829436302, 0.07658310234546661, -0.07591474056243896, 0.02063458040356636, 0.03785603865981102, -4.542695023701526e-05, 0.039139535278081894, 0.034109797328710556, -0.02708895318210125, -0.10462252050638199, 0.05202745646238327, -0.1600441187620163, -0.020805248990654945, -0.04254060983657837, 0.05350736901164055, -0.001641566

In [37]:
# Insert points into the collection
client.upsert(collection_name="company_benefits", points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [39]:
# Query search
query_vector = embedding_model.encode(["stock purchase"])[0]

# Perform the search
search_result = client.query_points(
    collection_name="company_benefits",
    query=query_vector,
    limit=2
)

print(search_result)

points=[ScoredPoint(id=3, version=0, score=0.5541204358808369, payload={'text': 'The Employee Stock Purchase Plan (ESPP) allows employees to purchase company \n        stock at a discounted rate. Employees can contribute up to 10% of their salary to buy shares \n        at a 15% discount. This plan is designed to encourage long-term investment in the company \n        and align employee interests with company success.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=0, score=0.12712619729047137, payload={'text': 'We offer a Professional Development Fund of $2,000 per year for each employee. \n        This fund can be used for attending conferences, enrolling in courses, obtaining certifications, \n        and purchasing educational materials. Employees are encouraged to discuss their development \n        plans with their managers to align with career goals.'}, vector=None, shard_key=None, order_value=None)]


## Pinecone